In [2]:
import pandas as pd

In [3]:
pd.read_pickle('../../outputs/stories_with_keywords_train.pkl').tail()

,text,Keyword 1,Keyword 2,Keyword 3,Keyword 4,Keyword 5
2119714,"Once upon a time, in a small town, there lived...",tim,club,friends,day,boss
2119715,"Once upon a time, there was a little boy named...",tim,ice,quit,time,wanted
2119716,"Once upon a time, there was a big tree. Under ...",tim,spread,food,judge,owl
2119717,"Once upon a time, there was a little girl name...",mia,monster,room,messy,clean
2119718,"Once upon a time, there was an adorable little...",named,kitty,car,park,upon


In [ ]:
import pandas as pd
import numpy as np
import ace_tools as tools

# Define the structure of the DataFrame
columns = [
    "Global Career Band", "Work Location Region Name", "Country", "Currency", "FX Rate",
    "Fixed Roll Up", "Total FTE used in FP", "Average Fixed Pay in USD", "Other Cost in %",
    "Average other cost in USD", "Total USD excluding VP", "Average Fixed Pay in Local Currency",
    "Average other Cost in Local Currency", "Total in Local Currency Excluding VP",
    "VP Roll Up", "Total FTE used in VP", "Average Variable Pay in USD",
    "Total in USD Currency including VP", "Average Variable Pay in Local Currency",
    "Total in Local Currency including Variable Pay", "Team", "Region Team"
]

# Dummy DataFrame for demonstration
df = pd.DataFrame(columns=columns)

# Define expected values for validation
expected_career_bands = {"MD", "3", "4", "5", "6", "7", "8"}
expected_regions = {"Asia", "Europe", "Latin America", "Middle East", "North America"}
expected_region_teams = {
    "Asia", "Canada", "Europe", "GSC & Tech", "Latin America",
    "Middle East", "United Kingdom", "United States"
}
expected_countries = {
    "Algeria", "Australia", "Bahrain", "Bangladesh", "Belgium", "Bermuda", "Brazil", "Canada", "China",
    "Czech Republic", "Egypt", "France", "Germany", "Hong Kong", "India", "Indonesia", "Ireland", "Italy",
    "Israel", "Japan", "Korea Republic of", "Kuwait", "Lebanon", "Luxembourg", "Macau", "Malaysia", "Malta",
    "Mauritius", "Mexico", "Netherlands", "New Zealand", "Oman", "Philippines", "Poland", "Qatar",
    "Saudi Arabia", "Singapore", "South Africa", "Spain", "Sri Lanka", "Sweden", "Switzerland", "Taiwan",
    "Thailand", "Turkiye", "United Arab Emirates", "United States", "Uruguay", "Viet Nam"
}
expected_roll_up_values = {0, 1, 2, 3}

# Initialize validation issue list
validation_issues = []

# 1. Null Value Checks
nulls = df.isnull().sum()
for col, count in nulls.items():
    if col != "Team" and count > 0:
        validation_issues.append(f"Column '{col}' has {count} missing values.")

# 2. Categorical Column Validation
def validate_categorical(col_name, expected_values):
    invalid = df[~df[col_name].isin(expected_values)][col_name]
    if not invalid.empty:
        validation_issues.append(
            f"Invalid values in column '{col_name}': {invalid.unique().tolist()}"
        )

validate_categorical("Global Career Band", expected_career_bands)
validate_categorical("Work Location Region Name", expected_regions)
validate_categorical("Country", expected_countries)
validate_categorical("Fixed Roll Up", expected_roll_up_values)
validate_categorical("VP Roll Up", expected_roll_up_values)
validate_categorical("Region Team", expected_region_teams)

# 3. FX Rate and Pay Cross-Validation
def validate_fx_consistency(row):
    if row["FX Rate"] and row["Average Fixed Pay in Local Currency"]:
        expected_usd = row["Average Fixed Pay in Local Currency"] / row["FX Rate"]
        if not np.isclose(row["Average Fixed Pay in USD"], expected_usd, atol=10):
            return False
    return True

if not df.empty:
    inconsistent_fx = df[~df.apply(validate_fx_consistency, axis=1)]
    if not inconsistent_fx.empty:
        validation_issues.append(f"{len(inconsistent_fx)} rows have FX inconsistencies.")

# 4. Logical Consistency on Roll Up and Pay
fp_zero = df[(df["Fixed Roll Up"] == 0) & (df["Average Fixed Pay in USD"] > 0)]
vp_zero = df[(df["VP Roll Up"] == 0) & (df["Average Variable Pay in USD"] > 0)]

if not fp_zero.empty:
    validation_issues.append(f"{len(fp_zero)} rows have 'Fixed Roll Up' = 0 but non-zero fixed pay.")

if not vp_zero.empty:
    validation_issues.append(f"{len(vp_zero)} rows have 'VP Roll Up' = 0 but non-zero variable pay.")

# 5. Other Cost % Valid Range
invalid_cost_percent = df[(df["Other Cost in %"] < 0) | (df["Other Cost in %"] > 100)]
if not invalid_cost_percent.empty:
    validation_issues.append(f"{len(invalid_cost_percent)} rows have 'Other Cost in %' outside 0-100 range.")

# 6. Outlier Detection (Z-score method)
def detect_outliers(col_name):
    if df[col_name].dtype.kind in 'fi':  # float or int
        z_scores = (df[col_name] - df[col_name].mean()) / df[col_name].std()
        return df[np.abs(z_scores) > 3]
    return pd.DataFrame()

outlier_columns = ["FX Rate", "Average Fixed Pay in USD", "Average Variable Pay in USD"]
for col in outlier_columns:
    outliers = detect_outliers(col)
    if not outliers.empty:
        validation_issues.append(f"{len(outliers)} outliers detected in column '{col}'.")

# Display the validation issues
issues_df = pd.DataFrame(validation_issues, columns=["Issues"])
tools.display_dataframe_to_user(name="Validation Issues", dataframe=issues_df)
